# 01 — Data audit

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = (ROOT / "../data/raw").resolve()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Research parameters
REBALANCE_FREQUENCY = "ME"
VOLATILITY_WINDOW = 60
ANNUAL_VOLATILITY_TARGET = 0.10
LEVERAGE_CAP = 4.0
LONG_FRACTION = 0.30
SHORT_FRACTION = 0.30
TRANSACTION_COST_BPS = 5.0
MINIMUM_TRAINING_MONTHS = 60
PROBABILITY_THRESHOLD = 0.50
RANDOM_STATE = 42

print("Data directory:", DATA_DIR)

from src.data_loader import inventory_data, read_parquet, discover_fx_columns

Data directory: C:\Users\vidhi\OneDrive\Desktop\Project lab\Summer\FX_Carry_26_Summer_PL\data\raw


In [2]:
files,columns=inventory_data(DATA_DIR)
display(files)
display(columns.head(100))
files.to_csv(OUTPUT_DIR/'data_inventory.csv',index=False)
columns.to_csv(OUTPUT_DIR/'column_inventory.csv',index=False)

,file,suffix,size_mb,rows_loaded,columns,status
0,download_coverage_summary.csv,.csv,0.000321,8,6,ok
1,em_fx_options_long.parquet,.parquet,16.652611,4847571,4,ok
2,em_fx_options_wide.parquet,.parquet,12.508799,5087,975,ok
3,em_fx_spot_forward_failures.csv,.csv,0.000893,8,4,ok
4,em_fx_spot_forward_long.parquet,.parquet,7.578670,1393179,4,ok
5,em_fx_spot_forward_missing_tickers.csv,.csv,0.000823,8,3,ok
6,em_fx_spot_forward_wide.parquet,.parquet,6.410471,5094,285,ok
7,em_interest_rates_failures.csv,.csv,0.000118,1,4,ok
8,em_interest_rates_long.parquet,.parquet,0.503102,134365,4,ok
9,em_interest_rates_missing_tickers.csv,.csv,0.000127,1,3,ok


,file,column
0,download_coverage_summary.csv,group
1,download_coverage_summary.csv,requested
2,download_coverage_summary.csv,fetched
3,download_coverage_summary.csv,missing
4,download_coverage_summary.csv,extra
...,...,...
95,em_fx_options_wide.parquet,USDCNH10B3M Curncy__PX_BID
96,em_fx_options_wide.parquet,USDCNH10B3M Curncy__PX_LAST
97,em_fx_options_wide.parquet,USDCNH10B6M Curncy__PX_ASK
98,em_fx_options_wide.parquet,USDCNH10B6M Curncy__PX_BID


In [3]:
FILES = {
    "em_spot_forward": "em_fx_spot_forward_wide.parquet",
    "g10_spot_forward": "g10_fx_spot_forward_wide.parquet",
    "em_interest_rates": "em_interest_rates_wide.parquet",
    "g10_interest_rates": "g10_interest_rates_wide.parquet",
    "em_onshore_rates": "em_onshore_rates_wide.parquet",
    "g10_rates_gaps": "g10_rates_gaps_wide.parquet",
    "em_risk": "em_risk_wide.parquet",
    "global_risk": "global_risk_wide.parquet",
    "macro_market_proxies": "macro_market_proxies_wide.parquet",
    "usd_riskfree": "usd_riskfree_wide.parquet",
}

for key, filename in FILES.items():
    path = DATA_DIR / filename
    print("\n", key, "->", filename)
    if path.exists():
        df = read_parquet(path)
        print(df.shape, df.index.min(), df.index.max())
        print(df.columns[:20].tolist())
    else:
        print("MISSING")



 em_spot_forward -> em_fx_spot_forward_wide.parquet
(5094, 285) 2007-01-01 00:00:00 2026-06-30 00:00:00
['BCN12M Curncy__PX_ASK', 'BCN12M Curncy__PX_BID', 'BCN12M Curncy__PX_LAST', 'BCN1M Curncy__PX_ASK', 'BCN1M Curncy__PX_BID', 'BCN1M Curncy__PX_LAST', 'BCN3M Curncy__PX_ASK', 'BCN3M Curncy__PX_BID', 'BCN3M Curncy__PX_LAST', 'BCN6M Curncy__PX_ASK', 'BCN6M Curncy__PX_BID', 'BCN6M Curncy__PX_LAST', 'BRL Curncy__PX_ASK', 'BRL Curncy__PX_BID', 'BRL Curncy__PX_LAST', 'CHN12M Curncy__PX_ASK', 'CHN12M Curncy__PX_BID', 'CHN12M Curncy__PX_LAST', 'CHN1M Curncy__PX_ASK', 'CHN1M Curncy__PX_BID']

 g10_spot_forward -> g10_fx_spot_forward_wide.parquet
(5087, 198) 2007-01-01 00:00:00 2026-06-30 00:00:00
['AUD Curncy__PX_ASK', 'AUD Curncy__PX_BID', 'AUD Curncy__PX_LAST', 'AUD12M Curncy__PX_ASK', 'AUD12M Curncy__PX_BID', 'AUD12M Curncy__PX_LAST', 'AUD1M Curncy__PX_ASK', 'AUD1M Curncy__PX_BID', 'AUD1M Curncy__PX_LAST', 'AUD1W Curncy__PX_ASK', 'AUD1W Curncy__PX_BID', 'AUD1W Curncy__PX_LAST', 'AUD3M Curn

In [4]:
# Automatic spot/forward discovery across the combined G10 + EM FX file
g10_fx = read_parquet(DATA_DIR / "g10_fx_spot_forward_wide.parquet")
em_fx = read_parquet(DATA_DIR / "em_fx_spot_forward_wide.parquet")
combined_fx = pd.concat([g10_fx, em_fx], axis=1)

spot_map, forward_map, inversion_map, diagnostics = discover_fx_columns(combined_fx)

display(diagnostics)
print(f"Matched currencies: {len(spot_map)}")
print("Currencies:", sorted(spot_map))

diagnostics.to_csv(OUTPUT_DIR / "fx_field_diagnostics.csv", index=False)


,currency,spot_column,forward_column,forward_root,invert_to_usd_per_fx,status
0,AUD,AUD Curncy__PX_LAST,AUD1M Curncy__PX_LAST,AUD,False,ok
1,BRL,BRL Curncy__PX_LAST,BCN1M Curncy__PX_LAST,BCN,True,ok
2,CAD,CAD Curncy__PX_LAST,CAD1M Curncy__PX_LAST,CAD,True,ok
3,CHF,CHF Curncy__PX_LAST,CHF1M Curncy__PX_LAST,CHF,True,ok
4,CLP,CLP Curncy__PX_LAST,CLN1M Curncy__PX_LAST,CLN,True,ok
5,CNH,CNH Curncy__PX_LAST,CNH1M Curncy__PX_LAST,CNH,True,ok
6,CNY,CNY Curncy__PX_LAST,CHN1M Curncy__PX_LAST,CHN,True,ok
7,COP,COP Curncy__PX_LAST,None,None,True,missing_1m_forward
8,DKK,DKK Curncy__PX_LAST,DKK1M Curncy__PX_LAST,DKK,True,ok
9,EUR,EUR Curncy__PX_LAST,EUR1M Curncy__PX_LAST,EUR,False,ok


Matched currencies: 26
Currencies: ['AUD', 'BRL', 'CAD', 'CHF', 'CLP', 'CNH', 'CNY', 'DKK', 'EUR', 'GBP', 'HUF', 'IDR', 'ILS', 'INR', 'JPY', 'MXN', 'MYR', 'NOK', 'NZD', 'PHP', 'PLN', 'SEK', 'SGD', 'THB', 'TRY', 'ZAR']
